# Lab 06 — Crack Detection with a CNN
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 6 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab06-cnn-crack-detection/Lab06_CNN_Crack_Detection.ipynb)

Inspectors spend thousands of hours looking for cracks in concrete. Today you build a crack detector in three steps: see how a computer stores a photo, slide small **filters** over it the way the first layer of a **convolutional neural network (CNN)** does, and then borrow a CNN that was trained on millions of everyday photos and teach it — with a few hundred examples — to recognize cracks. This is called **transfer learning**. A short last part uses the same neural-network ideas on a **time series**: forecasting next-hour traffic. Questions marked 🏠 may be finished at home before the deadline; sections marked *Optional* can be skipped.

### What you will be able to do
1. Describe a digital image as a grid of numbers (height × width × color channels).
2. Apply a 3×3 filter by hand and with code, and explain what a feature map and pooling show.
3. Use a pretrained CNN (ResNet-18) as a feature extractor and train a small classifier on top of it.
4. Evaluate the classifier with accuracy, a confusion matrix, precision, and recall, and inspect the hardest cases.
5. Forecast a time series with a sliding window and compare it with a simple baseline.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** this notebook runs faster on a GPU. In Colab: **Runtime → Change runtime type → T4 GPU → Save**. It still works on CPU, just more slowly.
**Data:** `concrete_cracks_sample.zip` — 600 crack and 600 no-crack photos of concrete surfaces, 227×227 pixels (Özgenel, 2019, CC BY 4.0). Part 4: `metro_traffic_volume.csv` (hourly traffic on I-94, CC BY 4.0).

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import glob, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — A photo is a grid of numbers (8 min)
Unzip the photos (about 7 MB). They land in two folders: `concrete_cracks/crack/` and `concrete_cracks/no_crack/`.

In [ ]:
zipfile.ZipFile(get_data("concrete_cracks_sample.zip")).extractall(".")
crack_files = sorted(glob.glob("concrete_cracks/crack/*.jpg"))
intact_files = sorted(glob.glob("concrete_cracks/no_crack/*.jpg"))
print(len(crack_files), "crack photos,", len(intact_files), "no-crack photos")

fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for ax, f in zip(axes[0], crack_files[:6]):
    ax.imshow(Image.open(f)); ax.set_title("crack"); ax.axis("off")
for ax, f in zip(axes[1], intact_files[:6]):
    ax.imshow(Image.open(f)); ax.set_title("no crack"); ax.axis("off")
plt.suptitle("Sample photos (each 227 × 227 pixels)")
plt.tight_layout(); plt.show()

`np.array(...)` turns a photo into a NumPy array. A color photo has three numbers per pixel — **red, green,
blue** — each a whole number from **0 (dark) to 255 (bright)**. `img[row, column]` reads one pixel; row 0 is the
top of the image.

In [ ]:
example = "concrete_cracks/crack/02327.jpg"
img = np.array(Image.open(example))
print("shape:", img.shape, "  <- (height, width, color channels)")
print("data type:", img.dtype, "  <- uint8 = whole numbers 0 to 255")
print("top-left pixel (red, green, blue):", img[0, 0])
print("numbers in this one photo:", img.size)

To keep things simple we convert to **grayscale**: one brightness number per pixel. The cell below finds
the crack in row 110 (the pixels darker than 110) and prints a small patch of brightness values across it: 5 rows ×
17 columns. `np.where(condition)` returns the positions where the condition is true.

In [ ]:
gray = np.array(Image.open(example).convert("L"), dtype=float)   # "L" = grayscale, 0 = black, 255 = white
dark = np.where(gray[110] < 110)[0]      # columns in row 110 darker than 110: the crack
col = int(dark.mean())                   # the middle of the crack
print(f"Brightness values in rows 108-112, columns {col - 8} to {col + 8}:")
print(gray[108:113, col - 8:col + 9].astype(int))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
axes[0].add_patch(plt.Rectangle((col - 8, 108), 17, 5, fill=False, edgecolor="red", linewidth=2))
axes[0].set_title("Grayscale photo (red box = patch)"); axes[0].axis("off")
im = axes[1].imshow(gray[100:121, col - 10:col + 11], cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Zoom: 21 × 21 pixels around the crack"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="brightness (0 = black, 255 = white)")
plt.tight_layout(); plt.show()

> 💡 **Hint:** NumPy arrays have `.mean()` and `.min()` methods, e.g. `gray.mean()`.

In [ ]:
# TODO: compare the brightness of the crack photo (gray) with the first no-crack photo.
gray_intact = np.array(Image.open(intact_files[0]).convert("L"), dtype=float)
crack_mean, crack_min = None, None      # mean and minimum brightness of gray
intact_mean, intact_min = None, None    # mean and minimum brightness of gray_intact
print(f"crack photo:    mean {crack_mean}, darkest {crack_min}")
print(f"no-crack photo: mean {intact_mean}, darkest {intact_min}")

**✍️ Question:** The crack is darker than the concrete around it. Would "average brightness" or "darkest pixel" be a reliable crack detector across hundreds of photos? Why or why not?

*Your answer:* 

## Part 2 — Convolution: sliding a filter over the image (20 min)

### 2.1 One filter, one position, by hand
A **filter** (or **kernel**) is a small grid of weights, here 3 × 3. **Convolution** slides it across the image.
At each position it multiplies the 9 pixel values by the 9 weights and adds them up — a weighted sum, just like a
neuron. The result at every position forms a new image called a **feature map**.

Below is a tiny 5 × 5 "photo": bright concrete (200) with a dark vertical line (60), and a **vertical-edge
filter**: negative weights on the left, positive on the right. It gives a big number where brightness changes
from left to right, and 0 where the surface is flat. **The sign only tells the direction of the change
(bright→dark vs. dark→bright); the size tells the edge strength.**

In [ ]:
tiny = np.array([[200, 200, 60, 200, 200],
                 [200, 200, 60, 200, 200],
                 [200, 200, 60, 200, 200],
                 [200, 200, 60, 200, 200],
                 [200, 200, 60, 200, 200]], dtype=float)    # bright surface with a dark vertical "crack"

vertical_edge = np.array([[-1, 0, 1],
                          [-2, 0, 2],
                          [-1, 0, 1]], dtype=float)

`tiny[0:3, 0:3]` cuts out rows 0–2 and columns 0–2 (the top-left 3 × 3 patch). For two arrays of the same shape, `*` multiplies matching entries, and `.sum()` adds everything up.

> 💡 **Hint:** `patch = tiny[0:3, 0:3]` and `value = (patch * vertical_edge).sum()`. By hand: each row gives −200 + 0 + 60, and the middle row counts twice.

In [ ]:
# TODO: apply the filter at the top-left position.
patch = None     # the top-left 3x3 patch of tiny
value = None     # multiply patch and vertical_edge entry by entry, then add up
print(patch)
print("Filter output at the top-left position:", value)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert patch is not None and value is not None, "Replace both None values in the cell above."
assert value == -560, "Expected -560: each row gives -200 + 0 + 60 = -140, and the rows count 1, 2, 1 times. Did you multiply patch * vertical_edge and then .sum()?"
print("Correct!")

The function below does the sliding for every position at once using SciPy. (Deep-learning libraries call
this operation *convolution*; mathematicians would call it *cross-correlation* because the filter is not flipped.
The idea is the same.) `mode="valid"` keeps only positions where the whole filter fits, so a 5 × 5 image gives a
3 × 3 feature map.

In [ ]:
from scipy.signal import correlate2d

def convolve(image, kernel):
    """Slide a small filter over a 2-D image and return the feature map."""
    return correlate2d(image, kernel, mode="valid")

print(convolve(tiny, vertical_edge))

**✍️ Question:** Explain the three columns of the feature map (−560, 0, 560). Where does the filter respond, and why is one side negative and the other positive?

*Your answer:* 

### 2.2 Filters on real photos
Now four hand-made filters on three real photos: a vertical crack, a horizontal crack, and no crack. We show the
**size** of each response (its absolute value): bright = strong response.

In [ ]:
filters = {
    "vertical edges":   vertical_edge,
    "horizontal edges": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float),
    "blur":             np.ones((3, 3)) / 9,
    "outline (all directions)": np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=float),
}

def load_gray(path):
    """Load a photo as a 2-D grayscale array with brightness from 0 to 1."""
    return np.array(Image.open(path).convert("L"), dtype=float) / 255

photos = {"vertical crack": "concrete_cracks/crack/02327.jpg",
          "horizontal crack": "concrete_cracks/crack/04682.jpg",
          "no crack": intact_files[0]}

fig, axes = plt.subplots(3, 5, figsize=(13, 8))
for r, (title, path) in enumerate(photos.items()):
    g = load_gray(path)
    axes[r, 0].imshow(g, cmap="gray"); axes[r, 0].set_title(title)
    for c, (name, k) in enumerate(filters.items(), start=1):
        fmap = convolve(g, k)
        if name == "blur":
            axes[r, c].imshow(fmap, cmap="gray")
        else:
            axes[r, c].imshow(np.abs(fmap), cmap="magma", vmin=0, vmax=1.5)
        axes[r, c].set_title(name, fontsize=10)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Feature maps: what each 3×3 filter responds to (bright = strong response)")
plt.tight_layout(); plt.show()

**✍️ Question:** Which filter lights up along the vertical crack, and which along the horizontal crack? What do the edge filters show on the no-crack photo?

*Your answer:* 

> 💡 **Hint:** Edge filters have negative weights on one side and positive weights on the other, adding up to 0. For a diagonal: put negatives in the lower-left corner and positives in the upper-right.

In [ ]:
# TODO: design your own 3x3 filter (for example, one that detects DIAGONAL edges) and show its feature map.
my_filter = np.array([[0, 0, 0],
                      [0, 1, 0],
                      [0, 0, 0]], dtype=float)     # change these 9 numbers (this one just copies the image)

g = load_gray(photos["vertical crack"])
plt.figure(figsize=(4, 4))
plt.imshow(np.abs(convolve(g, my_filter)), cmap="magma")
plt.title("My filter's feature map"); plt.axis("off"); plt.show()

### 2.3 Pooling: shrink the map, keep the strong responses
After a convolution, CNNs usually **pool**: split the feature map into 2 × 2 blocks and keep only the largest value
in each block. The map becomes half as wide and half as tall, but strong responses (like a crack edge) survive.
You don't need to follow the NumPy details of `max_pool` — just what it does.

In [ ]:
def max_pool(fmap, size=2):
    """Keep the largest value in each size x size block (the map shrinks by `size` in each direction)."""
    h, w = fmap.shape[0] // size * size, fmap.shape[1] // size * size
    blocks = fmap[:h, :w].reshape(h // size, size, w // size, size)
    return blocks.max(axis=(1, 3))

fmap = np.abs(convolve(load_gray(photos["vertical crack"]), vertical_edge))
pooled1 = max_pool(fmap)
pooled2 = max_pool(pooled1)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, m, t in zip(axes, [fmap, pooled1, pooled2], ["feature map", "after 1 pooling", "after 2 poolings"]):
    ax.imshow(m, cmap="magma", vmin=0, vmax=1.5); ax.set_title(f"{t}: {m.shape[0]} × {m.shape[1]}"); ax.axis("off")
plt.tight_layout(); plt.show()

**✍️ Question:** 🏠 *Finish at home if you run short of time.* What does pooling keep, and what does it throw away? Why might that be useful for detecting a crack anywhere in the photo?

*Your answer:* 

### 2.4 A hand-made crack detector
Can our edge filters alone sort the photos? For each photo we compute one **edge score**: the strength of its
strongest edges (the 99th percentile of the edge map — the value that 99% of pixels are below, which ignores a few
odd pixels). Then we pick a **threshold**: score above it → "crack".

First we split the photos into a **training set** (to choose the threshold) and a **test set** (to check it). We use
**300 photos of each class** to keep the run short on a CPU; on a GPU runtime you can set `N_PER_CLASS = 600` to use
all of them. `stratify=labels` makes a **stratified split**: both sets keep the same crack / no-crack mix
(here 50/50).

In [ ]:
from sklearn.model_selection import train_test_split

N_PER_CLASS = 300      # photos per class (max 600)
files = crack_files[:N_PER_CLASS] + intact_files[:N_PER_CLASS]
labels = np.array([1] * N_PER_CLASS + [0] * N_PER_CLASS)        # 1 = crack, 0 = no crack
files_train, files_test, y_train, y_test = train_test_split(
    files, labels, test_size=0.25, random_state=42, stratify=labels)
print(len(files_train), "training photos,", len(files_test), "test photos")

In [ ]:
def edge_score(path):
    """Strength of the strongest edges in a photo (99th percentile of the edge map)."""
    g = load_gray(path)
    gx = convolve(g, filters["vertical edges"])
    gy = convolve(g, filters["horizontal edges"])
    return np.percentile(np.sqrt(gx**2 + gy**2), 99)

t0 = time.time()
score_train = np.array([edge_score(f) for f in files_train])
score_test = np.array([edge_score(f) for f in files_test])
print(f"Edge scores computed in {time.time() - t0:.0f} s")

plt.figure(figsize=(7, 3.5))
bins = np.linspace(0, score_train.max(), 40)
plt.hist(score_train[y_train == 0], bins=bins, alpha=0.7, label="no crack", color="#378DBD")
plt.hist(score_train[y_train == 1], bins=bins, alpha=0.7, label="crack", color="#AB0520")
plt.xlabel("Edge score (strength of strongest edges, no units)")
plt.ylabel("Number of training photos")
plt.title("Hand-made feature: edge score by class")
plt.legend(); plt.show()

> 💡 **Hint:** Pick the x-value where the blue and red bars overlap least (somewhere between the two humps). `score_train > threshold` gives True/False for each photo; comparing it with `y_train` and taking `.mean()` gives the fraction correct. Try two or three thresholds.

In [ ]:
# TODO: look at the histogram and choose a threshold that separates the two classes as well as possible.
threshold = None    # a number on the x-axis of the histogram

acc_train_edge = ((score_train > threshold) == y_train).mean()
acc_test_edge = ((score_test > threshold) == y_test).mean()
print(f"Edge-score detector: train accuracy {acc_train_edge:.1%}, test accuracy {acc_test_edge:.1%}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert threshold is not None, "Replace None with a number read from the histogram's x-axis (for example 0.6)."
assert acc_test_edge > 0.8, "Your threshold can do better — look again where the two histograms overlap."
print("Good — a hand-made feature already works quite well.")

## Part 3 — Transfer learning with a pretrained CNN (20 min)

### 3.1 Load ResNet-18
**ResNet-18** is a CNN with 18 layers that have weights. It was trained on **ImageNet**: about 1.28 million everyday
photos in 1,000 categories (dogs, cars, bridges, ...). Its many filters were *learned*, not hand-made. We will use the
library **PyTorch** (`torch`) and its image add-on **torchvision**. You won't write PyTorch from scratch — just read
the comments to see what each line does. The first run downloads the model (about 45 MB). Model downloads happen on
Google's computers, not on your laptop or phone data plan.

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"   # use a GPU if Colab gave us one, otherwise the CPU
weights = ResNet18_Weights.DEFAULT        # the ImageNet-trained weights (downloads about 45 MB the first time)
model = resnet18(weights=weights)         # build the 18-layer network and load those weights
model.eval()                              # "use" mode, not training mode (some layers behave differently)
model.to(device)                          # move the model's numbers to the GPU or CPU
preprocess = weights.transforms()         # the same photo preparation used when ResNet was trained

n_params = sum(p.numel() for p in model.parameters())   # count every weight and bias
print(f"ResNet-18 has {n_params:,} parameters. Running on: {device}")
print(preprocess)

`preprocess` resizes each photo to 256 pixels, cuts out the central 224 × 224 square, turns the pixel values
into numbers from 0 to 1, and **normalizes** each color channel (subtracts the ImageNet average and divides by its
spread) — the same idea as `StandardScaler`.

### 3.2 What does the original ResNet think our photos are?
ImageNet has **no "crack" category**. Let's see what the network guesses anyway.

In [ ]:
categories = weights.meta["categories"]    # the 1,000 ImageNet class names

def imagenet_top3(path):
    """The 3 ImageNet categories ResNet-18 thinks most likely for one photo."""
    x = preprocess(Image.open(path).convert("RGB"))   # prepare the photo: tensor of shape (3, 224, 224)
    x = x.unsqueeze(0).to(device)                     # add a "batch" dimension: (1, 3, 224, 224)
    with torch.no_grad():                             # not training, so skip gradient bookkeeping (faster)
        probs = model(x).softmax(dim=1)[0]            # 1,000 scores -> probabilities that add up to 1
    top = probs.topk(3)                               # the 3 largest probabilities and their positions
    return [(categories[i], round(float(p), 2)) for p, i in zip(top.values, top.indices)]

for name, path in photos.items():
    print(f"{name:>16}: {imagenet_top3(path)}")

**✍️ Question:** What did ResNet guess for the crack photos? Why might those guesses make visual sense, even though they are wrong?

*Your answer:* 

### 3.3 Optional: look inside at the filters ResNet learned
*Skip this section if you are short on time — nothing later depends on it.* The very first layer, `model.conv1`, has 64 filters of size 7 × 7 with 3 color channels. Nobody designed them —
they were learned from ImageNet by gradient descent. Compare them with your hand-made 3 × 3 filters.

In [ ]:
w = model.conv1.weight.detach().cpu()      # detach: we only want the numbers; shape (64, 3, 7, 7)
print("conv1 weights:", tuple(w.shape), "= 64 filters × 3 colors × 7 × 7")

fig, axes = plt.subplots(4, 16, figsize=(12, 3.4))
for ax, f in zip(axes.flat, w):
    f = (f - f.min()) / (f.max() - f.min())         # rescale to 0-1 so we can show it as a color image
    ax.imshow(f.permute(1, 2, 0).numpy())           # (color, row, col) -> (row, col, color) for plotting
    ax.axis("off")
plt.suptitle("The 64 learned filters of ResNet-18's first layer")
plt.show()

In [ ]:
x = preprocess(Image.open(photos["vertical crack"]).convert("RGB")).unsqueeze(0).to(device)
with torch.no_grad():
    maps = model.conv1(x)[0].cpu()                 # run ONLY the first layer: 64 feature maps of 112 x 112
print("feature maps:", tuple(maps.shape))

fig, axes = plt.subplots(2, 8, figsize=(13, 4.4))
for i, ax in enumerate(axes.flat):
    ax.imshow(maps[i].numpy(), cmap="magma"); ax.set_title(f"filter {i}", fontsize=9); ax.axis("off")
plt.suptitle("First-layer feature maps for the vertical-crack photo (16 of 64)")
plt.tight_layout(); plt.show()

**✍️ Question:** (Optional) Describe two kinds of patterns the learned first-layer filters look for (look at their stripes and colors). How are they similar to your hand-made filters, and how are they different?

*Your answer:* 

### 3.4 Remove the last layer → a feature extractor
ResNet's final layer, `model.fc` (fc = *fully connected*), turns 512 numbers into 1,000 ImageNet scores. We replace
it with `Identity` — a layer that does nothing — so the network now outputs the **512 features** it computed for
each photo: a summary of edges, textures, and shapes. All 11 million weights stay **frozen** (unchanged).
(If you want to re-run section 3.2 later, re-run the cell in 3.1 first to get the full model back.)

In [ ]:
print("old final layer:", model.fc)
model.fc = torch.nn.Identity()       # remove the 1,000-class layer: output is now 512 numbers per photo

def extract_features(paths, batch_size=32):
    """Run photos through ResNet-18 (without its last layer). Returns an array of shape (n_photos, 512)."""
    feats = []
    with torch.no_grad():                                      # no training -> no gradients needed
        for start in range(0, len(paths), batch_size):         # 32 photos at a time ("a batch")
            batch_paths = paths[start:start + batch_size]
            batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in batch_paths])  # (32, 3, 224, 224)
            feats.append(model(batch.to(device)).cpu().numpy())   # (32, 512), back to NumPy on the CPU
    return np.concatenate(feats)                               # glue the batches together

t0 = time.time()
F_train = extract_features(files_train)
F_test = extract_features(files_test)
print("training features:", F_train.shape, " test features:", F_test.shape)
print(f"Feature extraction took {time.time() - t0:.0f} s on the {device}")

This is the slow step: every photo passes through 18 layers. On Colab's free CPU it takes a minute or
two for 600 photos; on a **T4 GPU** (Runtime → Change runtime type) it takes seconds. From here on everything is fast.

### 3.5 Train a small classifier "head"
Now we train a **logistic regression** (the classification model from Week 4) on the 512 features. It is a single
neuron: a weighted sum of 512 features plus a bias, passed through a sigmoid to give P(crack). So we train only
**513 parameters** and reuse 11 million. That is **transfer learning**.

> 💡 **Hint:** `head = LogisticRegression(max_iter=1000)`, then `head.fit(F_train, y_train)`, then `acc_test_cnn = head.score(F_test, y_test)`.

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: create LogisticRegression(max_iter=1000), fit it on (F_train, y_train), and compute the TEST accuracy.
head = None
acc_test_cnn = None      # hint: head.score(features, labels) returns the accuracy
print(f"ResNet-18 features + logistic regression: test accuracy {acc_test_cnn}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert head is not None and acc_test_cnn is not None, "Fit the head and compute the test accuracy."
assert acc_test_cnn > 0.9, "Expected well above 90% — did you fit on F_train and score on F_test?"
print("Correct!")

### 3.6 The confusion matrix
Accuracy is one number. A **confusion matrix** shows *which* mistakes happen. Rows = the actual class, columns = the
predicted class.
- **True positive (TP):** a crack, predicted crack. **False negative (FN):** a crack the model **missed**.
- **False positive (FP):** no crack, but flagged as crack (a false alarm). **True negative (TN):** correctly clear.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred_test = head.predict(F_test)
cm = confusion_matrix(y_test, pred_test, labels=[1, 0])     # order: crack first, then no crack
ConfusionMatrixDisplay(cm, display_labels=["crack", "no crack"]).plot(cmap="Blues", colorbar=False)
plt.title(f"Test set: {len(y_test)} photos, accuracy {acc_test_cnn:.1%}")
plt.xlabel("Predicted class"); plt.ylabel("Actual class")
plt.show()
print("TP =", cm[0, 0], " FN =", cm[0, 1], " FP =", cm[1, 0], " TN =", cm[1, 1])

### 3.7 Precision and recall
Two numbers describe how well the model handles the **crack** class:
- **Recall** = TP / (TP + FN): of all the **real cracks**, what fraction did the model **find**? Low recall means
  missed cracks.
- **Precision** = TP / (TP + FP): of all the photos the model **flagged** as cracks, what fraction **really are**
  cracks? Low precision means many false alarms.

In [ ]:
TP, FN, FP, TN = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
precision = TP / (TP + FP)      # of the flagged photos, the fraction that really show a crack
recall = TP / (TP + FN)         # of the real cracks, the fraction the model found
print(f"Our model: precision {precision:.1%}, recall {recall:.1%}")

Our model is (nearly) perfect on this easy, balanced test set. The Week 6 lecture used a less perfect crack classifier: **TP = 92, FN = 8, FP = 4, TN = 96** (200 photos).

> 💡 **Hint:** Copy the two formulas from the cell above and use the `_ex` names.

In [ ]:
# TODO: compute precision and recall for the lecture's example.
TP_ex, FN_ex, FP_ex, TN_ex = 92, 8, 4, 96
precision_ex = None
recall_ex = None
print(f"Lecture example: precision {precision_ex}, recall {recall_ex}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert precision_ex is not None and recall_ex is not None, "Replace both None values."
assert abs(precision_ex - 92 / 96) < 1e-9, "Precision = TP / (TP + FP): 92 flagged correctly out of 92 + 4 = 96 flagged."
assert abs(recall_ex - 0.92) < 1e-9, "Recall = TP / (TP + FN): 92 found out of 92 + 8 = 100 real cracks."
print("Correct! Precision 95.8%, recall 92.0%.")

**✍️ Question:** Suppose the lecture's model **screens** inspection photos: only the photos it flags are sent to an engineer. What happens to the 8 false negatives and to the 4 false positives? Why does **recall** matter more than precision for this kind of screening?

*Your answer:* 

### 3.8 Look at the hardest photos
`predict_proba` gives the model's probability of "crack" for each photo. Below are the 8 test photos where it was
**least sure** (probability closest to 50%), plus any mistakes. Titles in **red** are wrong.

In [ ]:
prob_crack = head.predict_proba(F_test)[:, 1]          # column 1 = probability of class 1 (crack)
wrong = np.where(pred_test != y_test)[0]
print(len(wrong), "mistakes out of", len(y_test), "test photos")

hardest = list(wrong) + [i for i in np.argsort(np.abs(prob_crack - 0.5)) if i not in wrong]
fig, axes = plt.subplots(2, 4, figsize=(12, 7.2))
for ax, i in zip(axes.flat, hardest[:8]):
    actual = "crack" if y_test[i] == 1 else "no crack"
    ax.imshow(Image.open(files_test[i]))
    ax.set_title(f"actual: {actual}\nP(crack) = {prob_crack[i]:.2f}",
                 color="#AB0520" if i in wrong else "black", fontsize=10)
    ax.axis("off")
plt.suptitle("The test photos the model was least sure about")
plt.tight_layout(h_pad=2.0); plt.show()

**✍️ Question:** 🏠 *Finish at home if you run short of time.* Describe the hardest photos. If there are no mistakes, describe the least-sure photos and what might confuse the model on a real bridge (thin hairline cracks, stains, shadows, joints, rough texture, the edges of the photo).

*Your answer:* 

### 3.9 Optional: how many labeled photos do you really need?
*Run these two cells if you have time.* Labeling photos takes an inspector's time. Because ResNet already "knows" edges and textures, the head may need only
a few examples. `balanced_subset(n)` picks n training photos: half cracks, half no cracks.

In [ ]:
def balanced_subset(n):
    """Positions of n training photos: n/2 cracks and n/2 no-cracks."""
    crack_idx = np.where(y_train == 1)[0][: n // 2]
    intact_idx = np.where(y_train == 0)[0][: n // 2]
    return np.concatenate([crack_idx, intact_idx])

In [ ]:
for n in [4, 10, 20, 50, 100, len(y_train)]:
    idx = balanced_subset(n)                                                  # n photos, half of each class
    clf = LogisticRegression(max_iter=1000).fit(F_train[idx], y_train[idx])   # train a new head on only those
    acc = clf.score(F_test, y_test)                                           # test on the same 150 test photos
    print(f"{len(idx):>4} training photos -> test accuracy {acc:.1%}")

In [ ]:
summary = pd.DataFrame({
    "method": ["Hand-made edge score + threshold", "ResNet-18 features + logistic regression"],
    "what was learned from our photos": ["1 threshold", "513 weights (11 million reused)"],
    "test accuracy": [f"{acc_test_edge:.1%}", f"{acc_test_cnn:.1%}"],
})
summary

**✍️ Question:** 🏠 *Finish at home if you run short of time.* Why can transfer learning reach high accuracy with only a few hundred labeled crack photos (or even fewer, if you ran §3.9)? Then give one reason this 99–100% accuracy might **not** hold on photos from a real bridge inspection.

*Your answer:* 

## Part 4 — Forecasting next-hour traffic with a sliding window (10 min)
CNNs suit grids (images). For **sequences** — sensor readings over time — a common idea is the **sliding window**:
use the last few hours to predict the next one. Recurrent networks (RNNs, LSTMs) read the window one step at a time
with a "memory"; here we keep it simple and let an MLP look at the whole window at once.

The data: hourly westbound traffic volume on Interstate 94 between Minneapolis and St. Paul (2012–2018).
`.asfreq("h")` makes one row per hour; hours with no record become `NaN` (missing).

In [ ]:
traffic = pd.read_csv(get_data("metro_traffic_volume.csv"), parse_dates=["date_time"])
volume = traffic.set_index("date_time")["traffic_volume"].asfreq("h")    # vehicles per hour
print(f"{len(volume):,} hours from {volume.index.min()} to {volume.index.max()}, "
      f"{volume.isna().sum():,} hours missing")

volume.loc["2017-10-02":"2017-10-08"].plot(figsize=(9, 3), color="#0C234B")
plt.ylabel("Traffic volume (vehicles/h)"); plt.xlabel("")
plt.title("One week on I-94 westbound (Mon 2 Oct – Sun 8 Oct 2017)")
plt.show()

`volume.shift(k)` slides the series down by k rows, so each row sees the value from **k hours earlier**.
We build a table where each row holds the previous 24 hours, the same hour one week earlier (168 hours), and the
**target** (the hour to predict). Rows with any missing hour are dropped.

For time series we **never shuffle** before splitting: we train on the past (July 2016 – 2017) and test on the future
(2018), like a real forecast. A **baseline** is a simple reference forecast that any model must beat; ours is
"traffic will be the same as at this hour last week".

In [ ]:
window = pd.DataFrame({f"lag_{k}h": volume.shift(k) for k in range(1, 25)})   # previous 24 hours
window["same_hour_last_week"] = volume.shift(168)                              # 7 days x 24 h earlier
window["target"] = volume
window = window.dropna()

train = window.loc["2016-07-01":"2017-12-31"]
test = window.loc["2018-01-01":]
inputs = [c for c in window.columns if c != "target"]
print(len(train), "training hours,", len(test), "test hours,", len(inputs), "inputs per hour")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Neural network on the sliding window (training takes about 10-30 s)
scaler = StandardScaler().fit(train[inputs])
mlp = MLPRegressor(hidden_layer_sizes=(32,), learning_rate_init=0.01, max_iter=60, random_state=42)
mlp.fit(scaler.transform(train[inputs]), train["target"])
pred_mlp = mlp.predict(scaler.transform(test[inputs]))
mae_mlp = mean_absolute_error(test["target"], pred_mlp)
print(f"MLP on 24-hour sliding window:  MAE = {mae_mlp:.0f} vehicles/h")
print(f"(average traffic in the test period: {test['target'].mean():.0f} vehicles/h)")

> 💡 **Hint:** Same function as for the MLP: `mean_absolute_error(actual, forecast)`, with the column `test["same_hour_last_week"]` as the forecast.

In [ ]:
# TODO: compute the baseline's MAE on the test hours: compare test["target"] with test["same_hour_last_week"].
mae_base = None
print(f"Same hour last week (baseline): MAE = {mae_base} vehicles/h")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert mae_base is not None, "Replace None with mean_absolute_error(...)."
assert 300 < mae_base < 400, "Expected a few hundred vehicles/h. Use test['target'] as the actual values and test['same_hour_last_week'] as the forecast."
print("Correct!")

In [ ]:
wk = slice("2018-03-05", "2018-03-11")
plt.figure(figsize=(10, 3.5))
plt.plot(test.loc[wk].index, test.loc[wk, "target"], label="actual", color="black")
plt.plot(test.loc[wk].index, test.loc[wk, "same_hour_last_week"], label="same hour last week", color="#378DBD", linestyle="--")
plt.plot(test.loc[wk].index, pred_mlp[test.index.get_indexer(test.loc[wk].index)], label="MLP forecast", color="#AB0520")
plt.ylabel("Traffic volume (vehicles/h)")
plt.ylim(0, 8500)                     # room for the legend above the peaks
plt.title("Next-hour forecasts, week of 5 March 2018 (test data)")
plt.legend(loc="upper left", ncol=3); plt.show()

**✍️ Question:** Which forecast is better, and by how much? Name one situation where the "same hour last week" baseline would fail badly, and whether the sliding-window model would do better there. (Hint: look at Monday 5 March in the plot; the data's `weather_main` column reports snow that afternoon.)

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** What is a feature map? Describe what the vertical-edge filter's feature map looked like for the vertical crack photo.

*Your answer:* 

**2.** Explain transfer learning in one or two sentences using today's crack detector.

*Your answer:* 

**3.** Your crack classifier scored 99–100% on our test photos. Give one reason it might score much lower on photos from a real bridge inspection.

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.